<a href="https://colab.research.google.com/github/TBGhorbanpour/Social-Awareness/blob/main/Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

  Using cached gensim-4.4.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
Using cached gensim-4.4.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (27.8 MB)


In [ ]:
!pip install hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.4 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9371 sha256=00d5e6f5daedd80cf4984aecb7b7a888d8e3184a7770639027d0dcbfc05555e3
  Stored in directory: /root/.cache/pip/wheels/be/12/c9/228313ff5cb722777830302f1d4136de4129932e6a0354c056
Successfully built flashtext


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Use the latest cleaned CSV from your previous step
INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_Sentiment_Output.csv'
#OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/MainDB_Topic Modeling.csv'
data = pd.read_csv(INPUT_CSV)

In [ ]:
data.shape

(75455, 25)

**Preprocess**

In [ ]:
import re
import unicodedata

def normalize_persian(text):
    if not isinstance(text, str):
        return ""

    # 1. Unicode normalization (fixes ﺩﻭﺳﺖ → دوست)
    text = unicodedata.normalize('NFKC', text)

    # 2. Arabic → Persian character mapping
    text = text.replace('ي', 'ی')   # Arabic Ya → Persian Ya
    text = text.replace('ك', 'ک')   # Arabic Kaf → Persian Kaf
    text = text.replace('ة', 'ه')   # Ta Marbuta → He
    text = text.replace('ؤ', 'و')
    text = text.replace('إ', 'ا')
    text = text.replace('أ', 'ا')
    text = text.replace('آ', 'ا')   # Optional: collapse Alef variants

    # 3. Fix half-spaces (Zero-Width Non-Joiner)
    # Replace regular spaces inside compound words with ZWNJ
    text = text.replace('\u200c', ' ')  # Normalize existing half-spaces
    text = re.sub(r'\s+', ' ', text)    # Collapse multiple spaces

    # 4. Remove URLs, mentions, hashtags symbols, emojis
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)       # Keep the word, remove the #
    text = re.sub(r'[^\w\s\u200c]', ' ', text)  # Remove punctuation

    # 5. Remove digits
    text = re.sub(r'[0-9۰-۹]', '', text)

    return text.strip()

**Topic Modeling using BOW**

In [ ]:
import pandas as pd
import re
import unicodedata
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from hazm import stopwords_list

# 1. Load Persian Stopwords
persian_stopwords = set(stopwords_list())

# 2. Add Custom Stopwords (Noise words identified from your 10-topic output)
custom_stopwords = {
    'و', 'در', 'به', 'از', 'که', 'می', 'این', 'است', 'را', 'با', 'های', 'ی',
    'برای', 'تا', 'اما', 'چه', 'آیا', 'هم', 'نه', 'بله', 'یک', 'دو', 'سه',
    'من', 'تو', 'او', 'ما', 'شما', 'آنها', 'کجا', 'کی', 'چرا', 'چگونه',
    # Added custom noise words to ensure cleaner topics:
    'دل', 'قیمت',
    'دنیا', 'گران',
     'ربط', 'قیمت', 'پول',
    'کشور', 'خبر', 'عزیز', 'دوست', 'یاد', 'کار', 'فیلم', 'موضوع' ,'خرید','شب','ماه' ,'قشنگ',' گوش','گل','زیبا','خوار','مرغ','ماده','جهان','زندگی','اروپا','ماشین','ادامه','هفته',
}

# Combine default and custom stopwords into a single list
final_stopwords = list(persian_stopwords.union(custom_stopwords))

# 3. Fast Normalization (No heavy lemmatization!)
def fast_normalize(text):
    if not isinstance(text, str): return ""
    # Fix Arabic/Persian characters
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('ي', 'ی').replace('ك', 'ک').replace('ة', 'ه')
    # Remove URLs, mentions, hashtags, numbers
    text = re.sub(r'http\S+|www\.\S+|@\S+|#', '', text)
    text = re.sub(r'[0-9۰-۹]', '', text)
    # Keep only letters and half-spaces
    text = re.sub(r'[^\w\s\u200c]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

print("Step 1: Cleaning text...")
clean_tweets = data['lemmatized_tweet'].astype(str).apply(fast_normalize)

print("Step 2: Vectorizing (C-optimized)...")
# CountVectorizer is written in C and is blazing fast
vectorizer = CountVectorizer(
    token_pattern=r'(?u)\b\w\w+\b', # Automatically ignores 1-letter words!
    stop_words=final_stopwords,       # <--- USES THE COMBINED STOPWORDS LIST
    max_features=10000
)
X = vectorizer.fit_transform(clean_tweets)

print("Step 3: Training LDA (Multi-core CPU)...")
# sklearn's LDA uses Cython and n_jobs for multi-core processing
lda_sklearn = LatentDirichletAllocation(
    n_components=10,
    random_state=100,
    n_jobs=-1,             # <--- USES ALL AVAILABLE CPU CORES
    max_iter=20,
    learning_method='online' # Much faster for large datasets
)
lda_sklearn.fit(X)

print("\n--- RESULTS ---")
# Print top 8 words for each topic
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_sklearn.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-8 - 1:-1]]
    print(f"Topic {topic_idx + 1}: {' | '.join(top_words)}")

Step 1: Cleaning text...
Step 2: Vectorizing (C-optimized)...


/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['آید', 'توان', 'تواند', 'توانند', 'رسد', 'رود', 'سال', 'نمی', 'گوش', 'گوید', 'گویند'] not in stop_words.
  warnings.warn(


Step 3: Training LDA (Multi-core CPU)...

--- RESULTS ---
Topic 1: سگ | ولگرد | حیوان | فرو | غذا | جمعیت | خانه | طبیعت
Topic 2: اب | دریاچه | ارومیه | سال | زمین | منبع | باران | طبیعی
Topic 3: زیست | محیط | جنگ | ایران | پسماند | مشکل | سازمان | منبع
Topic 4: خشکسالی | سال | گرد | غبار | استان | هوا | ایران | خوزستان
Topic 5: اعتراض | دوره | روستا | قانون | محیطب | شهرداری | گربه | تجمع
Topic 6: قحطی | ایران | گرسنه | فقر | غذا | اهواز | زنده | گندم
Topic 7: نان | ایرانی | دروغ | خانه | پوشش | خالی | روغن | برق
Topic 8: میانکاله | تالاب | شرایط | رئیس | منطقه | صدا | احیا | دولت
Topic 9: حمایت | الوده | جان | تولید | مرگ | گیاهی | زباله | ایران
Topic 10: زاینده | اصفهان | اب | خشک | کارون | شهر | تهران | ملت


In [ ]:
import gensim.corpora as corpora
from gensim.models import CoherenceModel

print("Calculating Coherence Scores...")

# 1. Extract the top 10 words for each topic from the sklearn model
feature_names = vectorizer.get_feature_names_out()
topics = []
for topic_idx, topic in enumerate(lda_sklearn.components_):
    # Get indices of top 10 words for this topic
    top_word_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_word_indices]
    topics.append(top_words)

# 2. Prepare texts and dictionary for Gensim
# Since clean_tweets is already normalized and space-separated, we can just split by space
texts = [doc.split() for doc in clean_tweets]
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# 3. Calculate C_v Coherence (Range: 0 to 1, higher is better)
# C_v compares the top words of a topic against the actual documents
coherence_cv = CoherenceModel(
    topics=topics,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)
cv_score = coherence_cv.get_coherence()

# 4. Calculate U_mass Coherence (Range: -14 to 0, closer to 0 is better)
# U_mass relies on document co-occurrence counts (requires the corpus)
coherence_umass = CoherenceModel(
    topics=topics,
    corpus=corpus,
    dictionary=dictionary,
    coherence='u_mass'
)
umass_score = coherence_umass.get_coherence()

print("="*40)
print(f"C_v Score:    {cv_score:.4f}  (Target: > 0.40 is good, > 0.50 is excellent)")
print(f"U_mass Score: {umass_score:.4f}  (Target: > -10.0 is acceptable, > -8.0 is good)")
print("="*40)

Calculating Coherence Scores...
C_v Score:    0.4176  (Target: > 0.40 is good, > 0.50 is excellent)
U_mass Score: -3.9660  (Target: > -10.0 is acceptable, > -8.0 is good)


In [ ]:
import pandas as pd
OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political__TopicModeling_Sentiment.csv'
# 1. Assign the dominant topic to each tweet in your dataframe
# (Topics will be numbered 1 to 10)
data['topic_id'] = lda_sklearn.transform(X).argmax(axis=1) + 1

# 2. Create the crosstab (Topic vs Sentiment)
crosstab_df = pd.crosstab(data['topic_id'], data['sentiment_label'])

# 3. Add a 'Total' column and row (Optional but highly recommended for analysis)
crosstab_df['Total_Tweets'] = crosstab_df.sum(axis=1)
crosstab_df.loc['Total'] = crosstab_df.sum(axis=0)

# 4. Save to CSV
crosstab_df.to_csv(OUTPUT_CSV, encoding='utf-8-sig')

print("CSV saved successfully ")
print("\nPreview of the crosstab:")
display(crosstab_df)

CSV saved successfully 

Preview of the crosstab:


sentiment_label,angry,delighted,furious,happy,neutral,Total_Tweets
topic_id,,,,,,
1,1033,546,2658,1471,1915,7623
2,2274,764,4746,2406,5129,15319
3,682,551,2800,1774,2677,8484
4,2815,698,3573,1823,2757,11666
5,342,216,857,467,1004,2886
6,783,704,4530,1788,1079,8884
7,457,422,1382,727,704,3692
8,399,206,728,586,1072,2991
9,545,437,1760,1040,1157,4939
